# Parameter Comparison: hydro-param vs NHM Reference

Compare pywatershed parameters generated by hydro-param against the NHM reference
dataset bundled with pywatershed (`drb_2yr/myparam.param`). Both cover the 765-HRU
Delaware River Basin domain.

**Expected differences:** Different source data vintages (NHM used older NLCD/soils),
different zonal stats engines, different DEM resolution.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pywatershed as pws
import xarray as xr

# Paths
REF_PARAM = Path(pws.__file__).parent / "data" / "drb_2yr" / "myparam.param"
HP_PARAM = Path("../pw-check/models/pywatershed/parameters.nc")

assert REF_PARAM.exists(), f"Reference not found: {REF_PARAM}"
assert HP_PARAM.exists(), f"hydro-param output not found: {HP_PARAM}"
print(f"Reference: {REF_PARAM}")
print(f"hydro-param: {HP_PARAM}")

## Load Parameters

In [ ]:
# Reference (PRMS .param format)
ref = pws.parameters.PrmsParameters.load(REF_PARAM)
ref_ds = ref.to_xr_ds()

# hydro-param (NetCDF)
hp = pws.parameters.PrmsParameters.from_netcdf(HP_PARAM, use_xr=True)
hp_ds = hp.to_xr_ds()

print(f"Reference: {dict(ref_ds.sizes)}, {len(ref_ds.data_vars)} variables")
print(f"hydro-param: {dict(hp_ds.sizes)}, {len(hp_ds.data_vars)} variables")

## Coordinate Alignment

Reference uses `nhm_id` coordinate (5307–7251), hydro-param uses `nhru` (5308–7252).
Both have 765 HRUs over the same DRB domain. The hydro-param IDs are offset by +1
from the reference because the NHM reference uses 1-based NHM IDs while hydro-param
uses the GeoFabric feature IDs. We align by position (same spatial ordering).

In [ ]:
print("Reference nhm_id:", ref_ds.nhm_id.values.min(), "–", ref_ds.nhm_id.values.max())
print("hydro-param nhru:", hp_ds.nhru.values.min(), "–", hp_ds.nhru.values.max())
print()

# Verify consistent offset
offset = hp_ds.nhru.values - ref_ds.nhm_id.values
assert np.all(offset == offset[0]), "ID offset is not constant — cannot align by position"
print(f"Constant ID offset: {offset[0]} (hydro-param = reference + {offset[0]})")
print(f"Both have {len(ref_ds.nhm_id)} HRUs — aligning by position.")

## Variable Inventory

In [ ]:
ref_vars = set(ref_ds.data_vars)
hp_vars = set(hp_ds.data_vars)
common = sorted(ref_vars & hp_vars)
only_ref = sorted(ref_vars - hp_vars)
only_hp = sorted(hp_vars - ref_vars)

print(f"Common variables: {len(common)}")
print(f"Only in reference: {len(only_ref)}")
print(f"Only in hydro-param: {len(only_hp)}")
print()
print("Only in reference:", only_ref)
print()
print("Only in hydro-param:", only_hp)

## Summary Statistics

For all common variables: mean, std, min, max for both datasets, plus mean absolute
difference and Pearson correlation.

In [ ]:
rows = []
for var in common:
    try:
        r = ref_ds[var].values.astype(float).ravel()
        h = hp_ds[var].values.astype(float).ravel()
    except (ValueError, TypeError):
        continue  # skip non-numeric
    if r.shape != h.shape:
        continue  # dimension mismatch (e.g. ndeplval differs)
    mask = np.isfinite(r) & np.isfinite(h)
    if mask.sum() < 2:
        continue
    r, h = r[mask], h[mask]
    corr = np.corrcoef(r, h)[0, 1] if np.std(r) > 0 and np.std(h) > 0 else np.nan
    rows.append(
        {
            "variable": var,
            "ref_mean": np.mean(r),
            "hp_mean": np.mean(h),
            "ref_std": np.std(r),
            "hp_std": np.std(h),
            "ref_min": np.min(r),
            "hp_min": np.min(h),
            "ref_max": np.max(r),
            "hp_max": np.max(h),
            "mae": np.mean(np.abs(r - h)),
            "corr": corr,
        }
    )

stats_df = pd.DataFrame(rows).set_index("variable").round(4)
print(f"{len(stats_df)} comparable variables")
stats_df.sort_values("corr", ascending=True)

## All Parameters: 1:1 Scatter Plots

Every comparable parameter plotted against the reference, sorted by correlation
(worst first). Parameters with zero variance (uniform defaults/seeds) are
collected in a separate summary table.

In [ ]:
# Split variables into plottable (spatial variation) vs uniform (constant in one or both)
plottable = []
uniform = []

for var in sorted(stats_df.index):
    row = stats_df.loc[var]
    if row["ref_std"] < 1e-12 or row["hp_std"] < 1e-12:
        uniform.append(var)
    else:
        plottable.append(var)

# Sort plottable by correlation (worst first for easy scanning)
plottable = sorted(plottable, key=lambda v: stats_df.loc[v, "corr"])

print(f"{len(plottable)} parameters with spatial variation (scatter plots below)")
print(f"{len(uniform)} uniform parameters (summary table follows)")

# --- Scatter grid ---
ncols = 6
nrows = (len(plottable) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows))
axes = np.atleast_2d(axes)

for idx, var in enumerate(plottable):
    ax = axes[idx // ncols, idx % ncols]
    r = ref_ds[var].values.astype(float).ravel()
    h = hp_ds[var].values.astype(float).ravel()
    mask = np.isfinite(r) & np.isfinite(h)
    r, h = r[mask], h[mask]
    ax.scatter(r, h, s=2, alpha=0.3)
    lo = min(r.min(), h.min())
    hi = max(r.max(), h.max())
    margin = (hi - lo) * 0.05 if hi > lo else 1.0
    ax.plot([lo - margin, hi + margin], [lo - margin, hi + margin], "r--", lw=0.8)
    corr = stats_df.loc[var, "corr"]
    color = "green" if corr > 0.8 else ("orange" if corr > 0.3 else "red")
    ax.set_title(f"{var}\nr={corr:.3f}", fontsize=8, color=color)
    ax.tick_params(labelsize=6)

for idx in range(len(plottable), nrows * ncols):
    axes[idx // ncols, idx % ncols].set_visible(False)

fig.suptitle(
    "All Parameters: hydro-param vs Reference (sorted by correlation, worst first)",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
plt.show()

## Uniform Parameters (No Spatial Variation)

Parameters where one or both datasets have zero standard deviation — typically
default constants or calibration seeds. Only the mean values differ.

In [ ]:
if uniform:
    udf = stats_df.loc[uniform, ["ref_mean", "hp_mean", "ref_std", "hp_std", "mae"]].copy()
    udf["match"] = np.where(udf["mae"] < 1e-6, "exact", "differs")
    display(udf.sort_values("mae", ascending=False))
    n_match = (udf["match"] == "exact").sum()
    print(f"\n{n_match}/{len(udf)} uniform parameters match exactly")
else:
    print("No uniform parameters found.")

## Spatial Difference Maps: Worst Parameters

Map the spatial pattern of (hydro-param - reference) for the 12 parameters with
the lowest correlation, to identify geographic patterns in the discrepancies.

In [ ]:
import geopandas as gpd

FABRIC_PATH = Path("../data/pywatershed_gis/drb_2yr/nhru.gpkg")
if not FABRIC_PATH.exists():
    print(f"Fabric not found at {FABRIC_PATH} — skipping spatial maps.")
    gdf = None
else:
    gdf = gpd.read_file(FABRIC_PATH)
    print(f"Loaded fabric: {len(gdf)} HRUs, CRS: {gdf.crs}")

In [ ]:
if gdf is not None:
    # Pick worst-correlation HRU-dimensioned parameters for spatial maps
    spatial_candidates = [
        v
        for v in plottable
        if v in ref_ds.data_vars
        and v in hp_ds.data_vars
        and ref_ds[v].dims == ("nhru",)
        or (ref_ds[v].dims == ("nhm_id",))
    ]
    # Take worst 12
    spatial_vars = spatial_candidates[:12]

    ncols = 4
    nrows = (len(spatial_vars) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
    axes = np.atleast_2d(axes)

    for idx, var in enumerate(spatial_vars):
        ax = axes[idx // ncols, idx % ncols]
        r = ref_ds[var].values.astype(float).ravel()
        h = hp_ds[var].values.astype(float).ravel()
        diff = h - r
        plot_gdf = gdf.copy()
        plot_gdf["diff"] = diff
        vmax = np.nanpercentile(np.abs(diff[np.isfinite(diff)]), 95)
        if vmax < 1e-10:
            vmax = 1.0
        plot_gdf.plot(
            column="diff",
            ax=ax,
            legend=True,
            cmap="RdBu_r",
            vmin=-vmax,
            vmax=vmax,
            legend_kwds={"shrink": 0.6},
        )
        corr = stats_df.loc[var, "corr"]
        ax.set_title(f"{var}\nr={corr:.3f}", fontsize=9)
        ax.set_axis_off()

    for idx in range(len(spatial_vars), nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.suptitle("Spatial Difference Maps (hp - ref): Worst Correlation Parameters", fontsize=13)
    fig.tight_layout()
    plt.show()
else:
    print("Skipping spatial maps (no fabric loaded).")

## Monthly Parameter Comparison

Parameters with a `nmonth` dimension (12 values per HRU): `jh_coef`, `dday_intcp`,
`jh_coef_hru`, `tmax_allrain_offset`, etc.

In [ ]:
monthly_vars = [
    v
    for v in common
    if "nmonth" in ref_ds[v].dims
    and "nmonth" in hp_ds[v].dims
    and ref_ds[v].shape == hp_ds[v].shape
]
print(f"Monthly parameters to compare: {monthly_vars}")

if monthly_vars:
    ncols = min(4, len(monthly_vars))
    nrows = (len(monthly_vars) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.atleast_2d(axes)

    months = np.arange(1, 13)
    for idx, var in enumerate(monthly_vars):
        ax = axes[idx // ncols, idx % ncols]
        # Mean across HRUs for each month
        r = ref_ds[var].values.astype(float)
        h = hp_ds[var].values.astype(float)
        # Shape is (nmonth, nhru) or (nhru, nmonth) — find month axis
        month_axis = list(ref_ds[var].dims).index("nmonth")
        r_mean = np.nanmean(r, axis=1 - month_axis)  # mean over HRUs
        h_mean = np.nanmean(h, axis=1 - month_axis)
        r_std = np.nanstd(r, axis=1 - month_axis)
        h_std = np.nanstd(h, axis=1 - month_axis)

        ax.errorbar(months - 0.1, r_mean, yerr=r_std, fmt="o-", ms=4, label="Reference", capsize=2)
        ax.errorbar(
            months + 0.1, h_mean, yerr=h_std, fmt="s-", ms=4, label="hydro-param", capsize=2
        )
        ax.set_title(var)
        ax.set_xlabel("Month")
        ax.set_xticks(months)
        ax.legend(fontsize=7)

    for idx in range(len(monthly_vars), nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.tight_layout()
    plt.show()

## Triage Summary

Classify all parameters by agreement quality to identify refinement priorities.

In [ ]:
def classify(row):
    """Classify parameter agreement into triage categories."""
    corr = row["corr"]
    ref_range = row["ref_max"] - row["ref_min"]
    row["mae"] / ref_range if ref_range > 0 else np.nan

    if row["ref_std"] < 1e-12 and row["hp_std"] < 1e-12:
        return "uniform-match" if row["mae"] < 1e-6 else "uniform-differs"
    if row["ref_std"] < 1e-12 or row["hp_std"] < 1e-12:
        return "one-uniform"
    if np.isnan(corr):
        return "no-correlation"
    if corr > 0.9:
        return "good (r>0.9)"
    if corr > 0.7:
        return "fair (r>0.7)"
    if corr > 0.3:
        return "weak (r>0.3)"
    return "poor (r<0.3)"


stats_df["category"] = stats_df.apply(classify, axis=1)

print("=== Parameter Agreement Summary ===\n")
for cat in [
    "good (r>0.9)",
    "fair (r>0.7)",
    "weak (r>0.3)",
    "poor (r<0.3)",
    "one-uniform",
    "uniform-match",
    "uniform-differs",
    "no-correlation",
]:
    members = stats_df[stats_df["category"] == cat].index.tolist()
    if members:
        print(f"{cat}: {len(members)} parameters")
        for v in members:
            row = stats_df.loc[v]
            corr_str = f"r={row['corr']:.3f}" if not np.isnan(row["corr"]) else "r=NaN"
            print(f"  {v:30s}  ref={row['ref_mean']:10.4f}  hp={row['hp_mean']:10.4f}  {corr_str}")
        print()

# Color-coded summary bar
cats = stats_df["category"].value_counts()
fig, ax = plt.subplots(figsize=(10, 1.5))
colors = {
    "good (r>0.9)": "#2ecc71",
    "fair (r>0.7)": "#f1c40f",
    "weak (r>0.3)": "#e67e22",
    "poor (r<0.3)": "#e74c3c",
    "uniform-match": "#3498db",
    "uniform-differs": "#9b59b6",
    "one-uniform": "#95a5a6",
    "no-correlation": "#bdc3c7",
}
left = 0
for cat in colors:
    if cat in cats:
        ax.barh(0, cats[cat], left=left, color=colors[cat], label=f"{cat} ({cats[cat]})")
        left += cats[cat]
ax.set_yticks([])
ax.set_xlabel("Number of parameters")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_title("Parameter Agreement Classification")
fig.tight_layout()
plt.show()

## Forcing Comparison

Compare forcing files if available.

In [ ]:
HP_FORCING = Path("../pw-check/models/pywatershed/forcing")
REF_FORCING = REF_PARAM.parent

# Check what forcing files exist
hp_forcing_files = sorted(HP_FORCING.glob("*.nc")) if HP_FORCING.exists() else []
ref_forcing_files = sorted(REF_FORCING.glob("*.nc"))

print(f"hydro-param forcing: {[f.name for f in hp_forcing_files]}")
print(f"Reference forcing: {[f.name for f in ref_forcing_files]}")

In [ ]:
forcing_vars = ["prcp", "tmax", "tmin"]

for fvar in forcing_vars:
    hp_file = HP_FORCING / f"{fvar}.nc" if HP_FORCING.exists() else None
    ref_file = REF_FORCING / f"{fvar}.nc"
    if hp_file and hp_file.exists() and ref_file.exists():
        hp_f = xr.open_dataset(hp_file)
        ref_f = xr.open_dataset(ref_file)
        # Find overlapping time
        common_time = np.intersect1d(hp_f.time.values, ref_f.time.values)
        if len(common_time) > 0:
            # Determine HRU dimension name for each dataset
            hp_hru_dim = "nhru" if "nhru" in hp_f[fvar].dims else "nhm_id"
            ref_hru_dim = "nhru" if "nhru" in ref_f[fvar].dims else "nhm_id"
            hp_sel = hp_f[fvar].sel(time=common_time).mean(dim=hp_hru_dim)
            ref_sel = ref_f[fvar].sel(time=common_time).mean(dim=ref_hru_dim)
            fig, ax = plt.subplots(figsize=(10, 3))
            ref_sel.plot(ax=ax, label="Reference", alpha=0.7)
            hp_sel.plot(ax=ax, label="hydro-param", alpha=0.7)
            ax.set_title(f"{fvar}: domain-mean time series")
            ax.legend()
            plt.tight_layout()
            plt.show()
        else:
            print(f"{fvar}: no overlapping time steps")
        hp_f.close()
        ref_f.close()
    else:
        print(f"{fvar}: files not found in both locations")